# RES-142 — Qwen3-Reranker-0.6B Colab setup smoke (G4/A100 only)

**Preparation artifact; all cells are unexecuted and GPU execution is not approved.** This notebook contains only B1 setup and a two-pair synthetic smoke.

Colab's account UI displayed G4 and A100 as selectable, but no runtime was allocated and no GPU-specific CU rate or VRAM variant was visible. A rate probe therefore has its own approval gate: APPROVE COLAB RATE INSPECTION must name one GPU and a numeric maximum CU charge. After that approval only, the operator manually connects that GPU, reads the positive CU/hour rate in Resources, records it, and disconnects/deletes the runtime without running notebook cells or installing packages. If no positive rate is visible, stop.

B1 requires a separate APPROVE COLAB SETUP naming the same GPU and a numeric maximum CU charge. The observed rate is used only for a projection; Colab exposes no hard per-run CU cap, so neither the notebook timer nor projection guarantees a maximum charge. Every outcome requires evidence export and manual runtime disconnect/delete.

## 1. Operator parameters — closed by default

The first code cell prompts for both exact approval phrases, the rate-inspection GPU and CU maximum, the observed positive rate, the B1 GPU and CU maximum, and the exact preparation commit/notebook SHA-256 from the review packet. Blank or invalid values stop before dependency setup. The B1 GPU must match the separately approved rate probe.

This notebook does not allocate or change a GPU. The operator selects the approved GPU manually before running the first cell.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import shutil
import sys
import time
from pathlib import Path
from uuid import uuid4

PHASE = "B1_SYNTHETIC_SMOKE"
MAX_RUNTIME_SECONDS = 45 * 60
SERVER_STARTUP_TIMEOUT_SECONDS = 15 * 60
SCORE_TIMEOUT_SECONDS = 120
RUN_STARTED = time.monotonic()
RUN_DEADLINE = RUN_STARTED + MAX_RUNTIME_SECONDS
SETUP_AUTHORIZATION = input("Enter exact APPROVE COLAB SETUP or leave blank to stop: ").strip()
if SETUP_AUTHORIZATION != "APPROVE COLAB SETUP":
    raise RuntimeError("B1 is locked. Obtain APPROVE COLAB SETUP first.")
RATE_INSPECTION_AUTHORIZATION = input(
    "Enter exact APPROVE COLAB RATE INSPECTION or leave blank to stop: "
).strip()
if RATE_INSPECTION_AUTHORIZATION != "APPROVE COLAB RATE INSPECTION":
    raise RuntimeError("A separately approved rate inspection is required.")
RATE_INSPECTION_GPU = input("GPU named in rate-inspection approval (G4 or A100): ").strip()
OBSERVED_COMPUTE_UNITS_PER_HOUR = float(
    input("Positive CU/hour rate observed in Colab Resources: ").strip()
)
RATE_INSPECTION_MAX_COMPUTE_UNITS = float(
    input("Numeric rate-inspection CU maximum: ").strip()
)
TARGET_GPU = input("GPU named in B1 setup approval (G4 or A100): ").strip()
APPROVED_MAX_COMPUTE_UNITS = float(input("Numeric B1 CU maximum: ").strip())
PREPARATION_COMMIT_SHA = input("Full preparation artifact commit SHA: ").strip()
EXECUTION_NOTEBOOK_SHA256 = input("Notebook SHA-256 from the review packet: ").strip()

ALLOW_DATA_TRANSFER = False
PERSIST_TO_DRIVE = False
if PHASE != "B1_SYNTHETIC_SMOKE" or ALLOW_DATA_TRANSFER or PERSIST_TO_DRIVE:
    raise RuntimeError("This notebook only permits the isolated synthetic B1 smoke.")
if TARGET_GPU not in {"G4", "A100"} or RATE_INSPECTION_GPU != TARGET_GPU:
    raise RuntimeError("B1 must use the same explicitly approved G4 or A100 as the rate probe.")

EXPECTED_LOCK_SHA256 = "d8a87ad2f1f2a7faccfb96fee9fa859995f61caa7b8a7b845e268888c4c111dc"
EXPECTED_TEMPLATE_SHA256 = "bf7b104801a4e4ad6a147d732e2d5a837a36f708cdbd1b871b864979d3e10acd"
EXPECTED_SUPPORT_SHA256 = "ca3c631a03edca02b6d7576e0d14e5af2a8101b413dc866efbe3189a1158426e"
EXPECTED_UPLOADS = {
    "res142_b1_support.py",
    "res142-colab.lock.txt",
    "res142_qwen3_reranker.jinja",
}
MODEL_ID = "Qwen/Qwen3-Reranker-0.6B"
MODEL_SHA = "e61197ed45024b0ed8a2d74b80b4d909f1255473"
TOKENIZER_REVISION = MODEL_SHA
VLLM_VERSION = "0.31.0"
MAX_MODEL_LEN = 8192
CLIENT_BATCH_SIZE = 8
MAX_NUM_SEQS = 8
MAX_NUM_BATCHED_TOKENS = 16384
GPU_MEMORY_UTILIZATION = 0.50

ROOT = Path("/content/res142")
if ROOT.exists():
    raise RuntimeError("Refusing to reuse or overwrite an earlier /content/res142 directory.")
ROOT.mkdir(parents=True)
OUTPUT_DIR = ROOT / "b1-synthetic-smoke"
OUTPUT_DIR.mkdir()
TOOLS_DIR = ROOT / "uv-tools"
UV_BIN = TOOLS_DIR / "bin" / "uv"
VENV = ROOT / "venv"
VENV_PYTHON = VENV / "bin" / "python"
VLLM_BIN = VENV / "bin" / "vllm"
from google.colab import files

uploaded = files.upload()
if set(uploaded) != EXPECTED_UPLOADS:
    raise RuntimeError(f"Upload exactly these reviewed files: {sorted(EXPECTED_UPLOADS)}")
SUPPORT_PATH = ROOT / "res142_b1_support.py"
LOCK_PATH = ROOT / "res142-colab.lock.txt"
TEMPLATE_PATH = ROOT / "qwen3_reranker.jinja"
SUPPORT_PATH.write_bytes(uploaded["res142_b1_support.py"])
LOCK_PATH.write_bytes(uploaded["res142-colab.lock.txt"])
TEMPLATE_PATH.write_bytes(uploaded["res142_qwen3_reranker.jinja"])

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

if sha256_file(SUPPORT_PATH) != EXPECTED_SUPPORT_SHA256:
    raise RuntimeError("B1 support helper digest mismatch; stop.")
if sha256_file(LOCK_PATH) != EXPECTED_LOCK_SHA256:
    raise RuntimeError("Candidate lock digest mismatch; stop before dependency setup.")
if sha256_file(TEMPLATE_PATH) != EXPECTED_TEMPLATE_SHA256:
    raise RuntimeError("Qwen3 template digest mismatch; stop before dependency setup.")

shutil.copy2(SUPPORT_PATH, OUTPUT_DIR / SUPPORT_PATH.name)
shutil.copy2(LOCK_PATH, OUTPUT_DIR / LOCK_PATH.name)
shutil.copy2(TEMPLATE_PATH, OUTPUT_DIR / "qwen3_reranker.jinja")

sys.path.insert(0, str(ROOT))
from res142_b1_support import (
    build_execution_provenance,
    process_group_exists,
    remaining_seconds,
    run_subprocess,
    sanitize_output,
    terminate_process_group,
    validate_setup_authorization,
    write_failure_manifest,
    write_json_atomic,
)

EXECUTION_PROVENANCE = build_execution_provenance(
    preparation_commit_sha=PREPARATION_COMMIT_SHA,
    notebook_sha256=EXECUTION_NOTEBOOK_SHA256,
    template_sha256=EXPECTED_TEMPLATE_SHA256,
    lock_sha256=EXPECTED_LOCK_SHA256,
)
PROJECTED_B1_COMPUTE_UNITS = validate_setup_authorization(
    setup_authorization=SETUP_AUTHORIZATION,
    rate_inspection_authorization=RATE_INSPECTION_AUTHORIZATION,
    rate_inspection_gpu=RATE_INSPECTION_GPU,
    rate_inspection_max_compute_units=RATE_INSPECTION_MAX_COMPUTE_UNITS,
    target_gpu=TARGET_GPU,
    observed_compute_units_per_hour=OBSERVED_COMPUTE_UNITS_PER_HOUR,
    approved_max_compute_units=APPROVED_MAX_COMPUTE_UNITS,
    max_runtime_seconds=MAX_RUNTIME_SECONDS,
)
def require_time_budget() -> float:
    return remaining_seconds(RUN_DEADLINE)

def _artifact_hashes() -> dict[str, str | None]:
    paths = {
        "support": SUPPORT_PATH,
        "lock": LOCK_PATH,
        "template": TEMPLATE_PATH,
        "pip_freeze": OUTPUT_DIR / "pip-freeze.txt",
        "vllm_log": OUTPUT_DIR / "vllm.log",
    }
    return {
        name: sha256_file(path) if path.is_file() else None
        for name, path in sorted(paths.items())
    }

def record_failure(stage: str, error: BaseException) -> None:
    try:
        error_output = str(
            getattr(error, "stderr", None)
            or getattr(error, "output", None)
            or globals().get("LAST_SUBPROCESS_OUTPUT", "")
        )
        server_log = OUTPUT_DIR / "vllm.log"
        if server_log.is_file():
            sanitized_log = sanitize_output(
                server_log.read_text(encoding="utf-8", errors="replace"),
                limit=1_000_000,
            )
            server_log.write_text(sanitized_log, encoding="utf-8")
            error_output += "\n" + sanitized_log[-12000:]
        observations = {
            "target_gpu": TARGET_GPU,
            "observed_compute_units_per_hour": OBSERVED_COMPUTE_UNITS_PER_HOUR,
            "approved_max_compute_units": APPROVED_MAX_COMPUTE_UNITS,
            "kernel_python": sys.version.split()[0],
            "platform": platform.platform(),
            "gpu": globals().get("GPU"),
            "software": globals().get("SOFTWARE"),
            "dependency_check": globals().get("PIP_CHECK"),
        }
        manifest_path = OUTPUT_DIR / "manifest.json"
        if manifest_path.exists():
            manifest_path = OUTPUT_DIR / f"failure-{time.time_ns()}.json"
        write_failure_manifest(
            manifest_path,
            stage=stage,
            error=error,
            execution_provenance=EXECUTION_PROVENANCE,
            artifact_sha256=_artifact_hashes(),
            observations=observations,
            error_output=error_output,
            runtime_seconds=time.monotonic() - RUN_STARTED,
        )
        archive_path = Path(
            shutil.make_archive(
                str(ROOT / f"res142-b1-failure-{uuid4().hex}"), "zip", root_dir=OUTPUT_DIR
            )
        )
        print(json.dumps({"failure_manifest": str(manifest_path), "archive": str(archive_path)}))
        try:
            files.download(str(archive_path))
        except Exception as download_error:
            print(f"Automatic evidence download failed: {sanitize_output(download_error)}")
    except Exception as evidence_error:
        print(f"Could not create/export failure evidence: {sanitize_output(evidence_error)}")
    finally:
        print("After any outcome, disconnect and delete runtime in Runtime > Disconnect and delete runtime.")

print(json.dumps({
    "authorization": "B1 only",
    "target_gpu": TARGET_GPU,
    "observed_cu_per_hour": OBSERVED_COMPUTE_UNITS_PER_HOUR,
    "approved_max_cu": APPROVED_MAX_COMPUTE_UNITS,
    "projected_b1_cu": PROJECTED_B1_COMPUTE_UNITS,
    "projected_cu_note": "projected CU (not a guaranteed platform cap)",
    "runtime_deadline_seconds": MAX_RUNTIME_SECONDS,
}, indent=2))


## 2. Read-only runtime and accelerator check

This checks the accelerator already selected by the operator. It refuses an unexpected SKU, count, VRAM size, compute capability, or Linux driver. Any failure writes and exports a sanitized failure manifest.

In [ ]:
def run_readonly(command: list[str], *, timeout_seconds: float = 30) -> str:
    result = run_subprocess(command, deadline=RUN_DEADLINE, timeout_seconds=timeout_seconds)
    if result.returncode != 0:
        globals()["LAST_SUBPROCESS_OUTPUT"] = result.stderr or result.stdout
        raise RuntimeError(f"Read-only probe failed: {command[0]} ({result.returncode}).")
    return result.stdout.strip()

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

CURRENT_STAGE = "gpu_preflight"
try:
    require_time_budget()
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("This pinned vLLM profile requires Linux x86_64. Stop.")
    glibc_name, glibc_version = platform.libc_ver()
    if glibc_name != "glibc" or tuple(int(part) for part in glibc_version.split(".")[:2]) < (2, 31):
        raise RuntimeError(
            f"The vLLM 0.31.0 dependency wheel floor is glibc 2.31; "
            f"found {glibc_name} {glibc_version}. Stop."
        )
    gpu_rows = run_readonly([
        "nvidia-smi", "--query-gpu=name,compute_cap,memory.total,driver_version",
        "--format=csv,noheader,nounits",
    ]).splitlines()
    if len(gpu_rows) != 1:
        raise RuntimeError(f"Expected exactly one visible GPU; found {len(gpu_rows)}. Stop.")
    fields = [field.strip() for field in gpu_rows[0].split(",")]
    if len(fields) != 4:
        raise RuntimeError("Unexpected nvidia-smi identity row. Stop.")
    gpu_name, capability_text, memory_mib_text, driver_text = fields
    capability = tuple(int(part) for part in capability_text.split("."))
    memory_mib = int(memory_mib_text)
    driver = tuple(int(part) for part in driver_text.split("."))
    if driver < (575, 57, 8):
        raise RuntimeError(f"Driver {driver_text} is below the proposed CUDA 12.9 floor. Stop.")
    if TARGET_GPU == "G4":
        if "RTX PRO 6000" not in gpu_name or "Blackwell" not in gpu_name:
            raise RuntimeError(f"Requested G4 but observed {gpu_name!r}. No automatic fallback.")
        if capability != (12, 0) or memory_mib < 90000:
            raise RuntimeError(f"G4 identity/96 GB floor mismatch: {capability_text}, {memory_mib} MiB.")
    else:
        if "A100" not in gpu_name or capability != (8, 0):
            raise RuntimeError(f"Requested A100 but observed {gpu_name!r}. No automatic fallback.")
        if not (38000 <= memory_mib <= 82000):
            raise RuntimeError(f"A100 memory is not a recognized 40/80 GB SKU: {memory_mib} MiB.")
    GPU = {"requested_target": TARGET_GPU, "name": gpu_name, "compute_capability": list(capability), "memory_mib": memory_mib, "memory_bytes_reported": memory_mib * 1024 * 1024, "driver_version": driver_text, "observed_compute_units_per_hour": OBSERVED_COMPUTE_UNITS_PER_HOUR, "approved_max_compute_units": APPROVED_MAX_COMPUTE_UNITS}
    GPU_MEMORY_OBSERVATIONS = []
    def gpu_memory_snapshot() -> dict[str, int]:
        row = run_readonly(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu", "--format=csv,noheader,nounits"])
        used_mib, total_mib, utilization_pct = (int(part.strip()) for part in row.split(","))
        if used_mib * 100 > total_mib * 80 or (total_mib - used_mib) * 100 < total_mib * 20:
            raise RuntimeError(f"GPU memory stop threshold exceeded ({used_mib}/{total_mib} MiB).")
        snapshot = {"used_mib": used_mib, "total_mib": total_mib, "utilization_pct": utilization_pct}
        GPU_MEMORY_OBSERVATIONS.append(snapshot)
        return snapshot
    gpu_memory_snapshot()
    print(json.dumps(GPU, indent=2))
except BaseException as exc:
    record_failure("gpu_preflight", exc)
    raise


## 3. Bootstrap uv, managed Python 3.12, and the hash-locked environment

The first cell accepts only the reviewed support module, lock, and Jinja file and verifies their SHA-256 values. This cell rechecks lock and template before installing the exact hash-locked graph into a separate Python 3.12 venv. Bootstrap failures write and export failure evidence.

In [ ]:
CURRENT_STAGE = "bootstrap"
try:
    UV_VERSION = "0.12.4"
    if TOOLS_DIR.exists() or (ROOT / "managed-python").exists():
        raise RuntimeError("Refusing to reuse partial uv/Python bootstrap state. Start with a fresh runtime.")
    TOOLS_DIR.mkdir(parents=True)
    bootstrap_command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-deps", "--only-binary=:all:", "--target", str(TOOLS_DIR), f"uv=={UV_VERSION}"]
    bootstrap = run_subprocess(bootstrap_command, deadline=RUN_DEADLINE, timeout_seconds=300)
    if bootstrap.returncode != 0 or not UV_BIN.is_file():
        globals()["LAST_SUBPROCESS_OUTPUT"] = bootstrap.stderr or bootstrap.stdout
        raise RuntimeError("Pinned uv bootstrap failed; stop without changing versions.")
    if run_readonly([str(UV_BIN), "--version"]) != f"uv {UV_VERSION}":
        raise RuntimeError("uv bootstrap version mismatch.")
    uv_env = os.environ.copy()
    uv_env["UV_PYTHON_INSTALL_DIR"] = str(ROOT / "managed-python")
    uv_env["UV_PYTHON_PREFERENCE"] = "only-managed"
    python_install = run_subprocess([str(UV_BIN), "python", "install", "3.12"], env=uv_env, deadline=RUN_DEADLINE, timeout_seconds=900)
    require_time_budget()
    if python_install.returncode != 0:
        globals()["LAST_SUBPROCESS_OUTPUT"] = python_install.stderr or python_install.stdout
        raise RuntimeError("Could not provision managed Python 3.12; stop.")
    venv_create = run_subprocess([str(UV_BIN), "venv", "--python", "3.12", str(VENV)], env=uv_env, deadline=RUN_DEADLINE, timeout_seconds=300)
    require_time_budget()
    if venv_create.returncode != 0:
        globals()["LAST_SUBPROCESS_OUTPUT"] = venv_create.stderr or venv_create.stdout
        raise RuntimeError("Could not create the isolated Python 3.12 venv; stop.")
    install_started = time.perf_counter()
    install_command = [str(UV_BIN), "pip", "sync", "--python", str(VENV_PYTHON), "--torch-backend", "cu129", "--only-binary", ":all:", "--require-hashes", "--strict", str(LOCK_PATH)]
    install = run_subprocess(install_command, env=uv_env, deadline=RUN_DEADLINE, timeout_seconds=1800)
    require_time_budget()
    if install.returncode != 0:
        globals()["LAST_SUBPROCESS_OUTPUT"] = install.stderr or install.stdout
        raise RuntimeError("Hash-locked binary-wheel sync failed; stop without changing pins or GPU.")
    install_seconds = time.perf_counter() - install_started
    pip_check = run_subprocess([str(UV_BIN), "pip", "check", "--python", str(VENV_PYTHON)], env=uv_env, deadline=RUN_DEADLINE, timeout_seconds=60)
    PIP_CHECK = pip_check.stdout.strip() or "no broken requirements"
    if pip_check.returncode != 0:
        globals()["LAST_SUBPROCESS_OUTPUT"] = pip_check.stderr or pip_check.stdout
        raise RuntimeError("Dependency check failed; do not smoke or resolve by upgrading packages.")
    pip_freeze = run_readonly([str(UV_BIN), "pip", "freeze", "--python", str(VENV_PYTHON)])
    PIP_VERSION = run_readonly([str(VENV_PYTHON), "--version"])
    (OUTPUT_DIR / "pip-freeze.txt").write_text(pip_freeze + "\n", encoding="utf-8")
    print(f"Install seconds: {install_seconds:.3f}; kernel Python: {sys.version.split()[0]}; isolated interpreter: {PIP_VERSION}; full package freeze captured.")
    print(PIP_CHECK)
except BaseException as exc:
    record_failure("bootstrap", exc)
    raise


## 4. Validate CUDA/PyTorch, package versions, and BF16 support

No model weights have been loaded yet. A CUDA/PyTorch mismatch or package drift is a hard stop with a failure manifest and best-effort evidence download.

In [ ]:
CURRENT_STAGE = "runtime_verification"
try:
    runtime_probe = r'''import importlib.metadata as md, json, platform, sys, torch
names = ["vllm", "numba", "torch", "torchvision", "torchaudio", "transformers", "huggingface-hub", "tokenizers", "safetensors"]
versions = {name: md.version(name) for name in names}
props = torch.cuda.get_device_properties(0)
print(json.dumps({"python": sys.version.split()[0], "platform": platform.platform(), "glibc": platform.libc_ver(), "packages": versions, "cuda_runtime": torch.version.cuda, "gpu_name": torch.cuda.get_device_name(0), "compute_capability": list(torch.cuda.get_device_capability(0)), "gpu_total_memory_bytes": props.total_memory, "bf16_supported": torch.cuda.is_bf16_supported()}))
'''
    probe = run_subprocess(
        [str(VENV_PYTHON), "-c", runtime_probe],
        deadline=RUN_DEADLINE,
        timeout_seconds=120,
    )
    if probe.returncode != 0:
        globals()["LAST_SUBPROCESS_OUTPUT"] = probe.stderr or probe.stdout
        raise RuntimeError("PyTorch/CUDA probe failed; stop.")
    SOFTWARE = json.loads(probe.stdout.strip())
    expected_versions = {
        "vllm": "0.31.0",
        "numba": "0.65.0",
        "torchvision": "0.28.0",
        "torchaudio": "2.11.0",
        "transformers": "5.10.4",
        "huggingface-hub": "1.31.0",
        "tokenizers": "0.22.1",
        "safetensors": "0.6.2",
    }
    for package, expected in expected_versions.items():
        if SOFTWARE["packages"][package].split("+", 1)[0] != expected:
            raise RuntimeError(
                f"Pinned package drift: {package}={SOFTWARE['packages'][package]}, expected {expected}."
            )
    if not SOFTWARE["packages"]["torch"].startswith("2.13.0") or not str(
        SOFTWARE["cuda_runtime"]
    ).startswith("12.9"):
        raise RuntimeError("Expected PyTorch 2.13.0 with CUDA 12.9. Stop.")
    if not SOFTWARE["bf16_supported"]:
        raise RuntimeError("The selected GPU/runtime does not report BF16 support. Stop.")
    if SOFTWARE["gpu_name"] != GPU["name"] or SOFTWARE["compute_capability"] != GPU[
        "compute_capability"
    ]:
        raise RuntimeError("GPU identity changed between nvidia-smi and PyTorch. Stop.")
    print(json.dumps(SOFTWARE, indent=2))
except BaseException as exc:
    record_failure("runtime_verification", exc)
    raise


## 5. Verify the uploaded template and prepare the private scoring server

The uploaded, hash-verified Jinja file is the only template authority. Its digest is checked again immediately before server startup; the notebook contains no template reconstruction. The server is bound to loopback only.

In [ ]:
CURRENT_STAGE = "template_verification"
try:
    TEMPLATE_SHA256 = sha256_file(TEMPLATE_PATH)
    EXPECTED_TEMPLATE_SHA256 = "bf7b104801a4e4ad6a147d732e2d5a837a36f708cdbd1b871b864979d3e10acd"
    if TEMPLATE_SHA256 != EXPECTED_TEMPLATE_SHA256:
        raise RuntimeError(f"Qwen3 score template digest mismatch: {TEMPLATE_SHA256}. Stop.")

    TASK_INSTRUCTION = "Given a web search query, retrieve relevant passages that answer the query"
    HF_HOME = ROOT / "hf-home"
    HF_HOME.mkdir(exist_ok=True)
    server_args = [
        str(VLLM_BIN), "serve", MODEL_ID,
        "--revision", MODEL_SHA, "--tokenizer-revision", TOKENIZER_REVISION,
        "--runner", "pooling", "--dtype", "bfloat16",
        "--max-model-len", str(MAX_MODEL_LEN), "--max-num-seqs", str(MAX_NUM_SEQS),
        "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--tensor-parallel-size", "1", "--seed", "0",
        "--hf_overrides", json.dumps({
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        }, separators=(",", ":")),
        "--chat-template", str(TEMPLATE_PATH),
        "--host", "127.0.0.1", "--port", "8000",
    ]
    server_env = os.environ.copy()
    server_env["HF_HOME"] = str(HF_HOME)
    server_env["HF_HUB_DISABLE_TELEMETRY"] = "1"
    SERVER_LOG = OUTPUT_DIR / "vllm.log"
    server_log_handle = SERVER_LOG.open("x", encoding="utf-8")
    print(json.dumps({"argv": server_args, "template_sha256": TEMPLATE_SHA256, "bind": "127.0.0.1:8000"}, indent=2))
except BaseException as exc:
    record_failure("template_verification", exc)
    raise


## 6. Synthetic scoring, evidence, and cleanup

Exactly two synthetic query-document pairs are scored. The vLLM process starts in its own process group. Cleanup sends TERM, then KILL if needed, and verifies the full group is gone. Startup, model verification, scoring, and cleanup failures all create evidence.

In [ ]:
import math
import urllib.error
import urllib.request

synthetic_query = "Which planet is called the Red Planet?"
synthetic_documents = ["Mars is called the Red Planet.", "Venus is Earth's twin."]
if len(synthetic_documents) != 2:
    raise RuntimeError("B1 is restricted to exactly two synthetic pairs.")
synthetic_input = {"query": synthetic_query, "documents": synthetic_documents}
input_sha256 = hashlib.sha256(json.dumps(synthetic_input, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()
process = None
execution_error = None
cleanup_error = None
CURRENT_STAGE = "vllm_startup"
started = time.monotonic()
startup_seconds = None
smoke_ms = None
scores = []
try:
    process = subprocess.Popen(server_args, stdout=server_log_handle, stderr=subprocess.STDOUT, env=server_env, start_new_session=True)
    deadline = min(RUN_DEADLINE, time.monotonic() + SERVER_STARTUP_TIMEOUT_SECONDS)
    while time.monotonic() < deadline:
        require_time_budget()
        gpu_memory_snapshot()
        if process.poll() is not None:
            raise RuntimeError("vLLM exited during startup. Stop; do not retry or change the profile.")
        try:
            with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=min(5, require_time_budget())) as response:
                if response.status == 200:
                    startup_seconds = time.monotonic() - started
                    break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            time.sleep(min(2, require_time_budget()))
    if startup_seconds is None:
        raise RuntimeError("vLLM did not become healthy within 15 minutes. Stop.")
    CURRENT_STAGE = "model_verification"
    model_snapshot = HF_HOME / "hub" / "models--Qwen--Qwen3-Reranker-0.6B" / "snapshots" / MODEL_SHA
    required_model_files = {"config.json", "tokenizer.json", "tokenizer_config.json", "model.safetensors"}
    if not model_snapshot.is_dir() or not required_model_files.issubset({path.name for path in model_snapshot.iterdir()}):
        raise RuntimeError("The local Hugging Face cache does not prove the pinned model snapshot/files. Stop.")
    hash_started = time.perf_counter()
    model_snapshot_files = [{"name": path.name, "size_bytes": path.stat().st_size, "sha256": sha256_file(path)} for path in sorted(model_snapshot.iterdir()) if path.is_file()]
    model_file_hashing_seconds = time.perf_counter() - hash_started
    require_time_budget()
    gpu_memory_snapshot()
    require_time_budget()
    CURRENT_STAGE = "scoring"
    payload = {
        "model": MODEL_ID,
        "queries": synthetic_query,
        "documents": synthetic_documents,
        "instruction": TASK_INSTRUCTION,
        "use_activation": True,
        "max_tokens_per_query": 1024,
        "max_tokens_per_doc": 6144,
    }
    request = urllib.request.Request("http://127.0.0.1:8000/score", data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json", "Accept": "application/json"}, method="POST")
    score_started = time.perf_counter()
    with urllib.request.urlopen(request, timeout=min(SCORE_TIMEOUT_SECONDS, require_time_budget())) as response:
        response_payload = json.loads(response.read().decode("utf-8"))
    smoke_ms = (time.perf_counter() - score_started) * 1000.0
    if process.poll() is not None:
        raise RuntimeError("vLLM exited after scoring; stop and report.")
    rows = response_payload.get("data")
    if response_payload.get("model") != MODEL_ID:
        raise RuntimeError("Scoring response model identity differs from the pinned model ID.")
    if not isinstance(rows, list) or len(rows) != 2:
        raise RuntimeError("Scoring response cardinality is not exactly two.")
    for row in rows:
        if not isinstance(row, dict) or isinstance(row.get("index"), bool) or row.get("index") not in {0, 1}:
            raise RuntimeError("Scoring response has an invalid/foreign index.")
        value = row.get("score")
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(float(value)):
            raise RuntimeError("Scoring response has a non-finite score.")
        scores.append({"synthetic_candidate_position": row["index"], "score": float(value)})
    if {row["synthetic_candidate_position"] for row in scores} != {0, 1}:
        raise RuntimeError("Scoring response indices are not a bijection of the two inputs.")
    gpu_memory_snapshot()
    require_time_budget()
except BaseException as exc:
    execution_error = exc
finally:
    cleanup_error = None
    if process is not None:
        try:
            terminate_process_group(process)
            if process_group_exists(process.pid):
                raise RuntimeError("A vLLM worker survived process-group cleanup.")
        except BaseException as exc:
            cleanup_error = exc
    try:
        server_log_handle.close()
    except BaseException as exc:
        if cleanup_error is None:
            cleanup_error = exc

if execution_error is not None:
    record_failure(CURRENT_STAGE, execution_error)
    if cleanup_error is not None:
        record_failure("process_cleanup", cleanup_error)
    raise execution_error
if cleanup_error is not None:
    record_failure("process_cleanup", cleanup_error)
    raise cleanup_error

try:
    def host_memory_snapshot() -> dict[str, int]:
        values = {}
        with Path("/proc/meminfo").open(encoding="utf-8") as handle:
            for line in handle:
                key, raw, *_ = line.split()
                if key in {"MemTotal:", "MemAvailable:"}:
                    values[key[:-1]] = int(raw) * 1024
        return values

    scores_path = OUTPUT_DIR / "synthetic-scores.json"
    scores_bytes = json.dumps(scores, sort_keys=True, separators=(",", ":")).encode("utf-8") + b"\n"
    scores_path.write_bytes(scores_bytes)
    server_log_bytes = SERVER_LOG.read_bytes() if SERVER_LOG.is_file() else b""
    SERVER_LOG.write_text(sanitize_output(server_log_bytes, limit=1_000_000), encoding="utf-8")
    template_bytes = TEMPLATE_PATH.read_bytes()
    shutil.copy2(LOCK_PATH, OUTPUT_DIR / "res142-colab.lock.txt")
    shutil.copy2(TEMPLATE_PATH, OUTPUT_DIR / "qwen3_reranker.jinja")
    shutil.copy2(SUPPORT_PATH, OUTPUT_DIR / "res142_b1_support.py")
    artifact_files = {
        name: {"size_bytes": path.stat().st_size, "sha256": sha256_file(path)}
        for name, path in sorted({
            "pip-freeze.txt": OUTPUT_DIR / "pip-freeze.txt",
            "synthetic-scores.json": scores_path,
            "vllm.log": SERVER_LOG,
            "qwen3_reranker.jinja": OUTPUT_DIR / "qwen3_reranker.jinja",
            "res142-colab.lock.txt": OUTPUT_DIR / "res142-colab.lock.txt",
            "res142_b1_support.py": OUTPUT_DIR / "res142_b1_support.py",
        }.items())
    }
    manifest = {
        "manifest_revision": "res142-colab-b1-v2",
        "status": "success",
        "phase": PHASE,
        "stop_status": "b1-synthetic-smoke-completed",
        "preparation_base_git_sha": "67edcc785af2084e2cc4d8c27b08bea43a5f4bd3",
        "execution_provenance": EXECUTION_PROVENANCE,
        "res142_implementation_git_sha": None,
        "evaluation_code_sha": None,
        "model": {"repo": MODEL_ID, "model_sha": MODEL_SHA, "tokenizer_revision": TOKENIZER_REVISION, "snapshot_commit_confirmed": model_snapshot.name, "snapshot_files": model_snapshot_files},
        "vllm_wheel_sha256": "5176ab746f4e736c4a354a11d0620abf0a941de287cad937ac34039293ca6f5f",
        "python": SOFTWARE["python"],
        "packages": SOFTWARE["packages"],
        "venv_python": PIP_VERSION,
        "uv_version": UV_VERSION,
        "pip_freeze_sha256": sha256_file(OUTPUT_DIR / "pip-freeze.txt"),
        "pip_check": pip_check.stdout.strip() or "no broken requirements",
        "gpu": GPU,
        "compute_units": {"rate_inspection_gpu": RATE_INSPECTION_GPU, "rate_inspection_maximum": RATE_INSPECTION_MAX_COMPUTE_UNITS, "observed_rate_per_hour": OBSERVED_COMPUTE_UNITS_PER_HOUR, "approved_maximum": APPROVED_MAX_COMPUTE_UNITS, "projected_b1_cu": PROJECTED_B1_COMPUTE_UNITS, "enforcement": "authorization only; Colab exposes no hard per-run CU cap"},
        "cuda_runtime": SOFTWARE["cuda_runtime"],
        "serving": {"argv": server_args, "endpoint": "http://127.0.0.1:8000/score", "template_sha256": hashlib.sha256(template_bytes).hexdigest()},
        "prompt": {"instruction": TASK_INSTRUCTION, "instruction_sha256": hashlib.sha256(TASK_INSTRUCTION.encode("utf-8")).hexdigest()},
        "inputs": {"kind": "synthetic-only", "pair_count": 2, "input_sha256": input_sha256, "dataset_sha256": None, "candidate_snapshot_sha256": None, "baseline_run_sha256": None},
        "results": {"score_count": len(scores), "scores_sha256": sha256_file(scores_path)},
        "timing": {"install_seconds": install_seconds, "server_startup_seconds": startup_seconds, "model_file_hashing_seconds": model_file_hashing_seconds, "smoke_request_ms": smoke_ms, "p50_ms": None, "p95_ms": None, "p99_ms": None},
        "cleanup": {"server_group_terminated": process is not None and not process_group_exists(process.pid), "server_returncode": process.returncode if process is not None else None, "runtime_disconnect_required": True},
        "resources": {"gpu_memory_observations": GPU_MEMORY_OBSERVATIONS, "gpu_memory_high_water_mib": max(row["used_mib"] for row in GPU_MEMORY_OBSERVATIONS), "host_memory": host_memory_snapshot(), "child_process_max_rss_kib": __import__("resource").getrusage(__import__("resource").RUSAGE_CHILDREN).ru_maxrss},
        "artifact_files": artifact_files,
        "runtime_seconds": time.monotonic() - RUN_STARTED,
    }
    manifest_path = OUTPUT_DIR / "manifest.json"
    write_json_atomic(manifest_path, manifest)
    write_json_atomic(OUTPUT_DIR / "manifest.sha256.json", {"manifest_sha256": sha256_file(manifest_path)})
    archive_path = Path(shutil.make_archive(str(ROOT / "res142-b1-synthetic-smoke"), "zip", root_dir=OUTPUT_DIR))
    archive_sha256 = sha256_file(archive_path)
    print(json.dumps(manifest, indent=2, sort_keys=True))
    print(json.dumps({"archive": str(archive_path), "archive_sha256": archive_sha256}, indent=2))
    from google.colab import files
    files.download(str(archive_path))
    print("B1 evidence exported. Disconnect and delete the runtime in Runtime > Disconnect and delete runtime.")
except BaseException as exc:
    record_failure("evidence_export", exc)
    raise


## 7. Persist the B1 smoke artifact and stop

After success or any failure, download evidence and use Runtime > Disconnect and delete runtime; B1 is not complete until the UI shows the runtime disconnected. The wall-clock deadline starts at the first code cell, so the operator timer must start at connect. Each blocking subprocess is bounded by the remaining deadline. CU limits are authorization ceilings, not platform-enforced caps.

After B1, report observed hardware, the separately observed rate, the authorized CU maximum, package freeze, model identity, timings, and artifact hashes. Do not add candidate inputs. The future 20-query SciFact pilot and full benchmark require separate approvals.